# Laue maps quickstart: from the scans of your experiment to orientation and grain maps

This notebook analyses a 2D map of Laue patterns (`amesh` scan) recorded at BM32, in 5 steps:

| step | what | result (in `PROCESSED_DATA`) |
|---|---|---|
| 1 | choose a map in the list of the scans of your experiment | |
| 2 | write the settings files of the analysis | `configs/peaksearch.yaml`, `configs/indexing.yaml` |
| 3 | **peak search**: find the Laue spots of every image | `corfiles/*.dat`, `*.cor` |
| 4 | **indexation**: find the orientation (and strain) of the crystal(s) at every point | `fitfiles_<material>/*.fit` |
| 5 | **maps**: quality, orientation (IPF colors), grains, misorientations (KAM, GROD) | figures |
| 6 | (option) the same analysis for a series of maps (e.g. temperatures) | |

**How to use it**
- edit only the **SETTINGS** cell, then run the cells one after the other (`Shift + Enter`)
- messages in **red** say what to change; in **green**, all is fine
- **running the notebook again is safe**: steps already done are not computed again (their results are reloaded), nothing is deleted. Images (`RAW_DATA`) are only read
- after a change of parameters (SETTINGS or YAML files), the steps concerned are computed again automatically; the previous results are kept in folders `*_previous_<date>`
- computations run on the ESRF cluster (SLURM, e.g. `magnifix` partition) when the notebook runs on https://jupyter-slurm.esrf.fr, otherwise on this computer with all its cpus

For more control: notebooks `../peaksearch/peaksearch_map.ipynb` and `../indexation/` (they use the settings files written here).

**This notebook is part of the LaueTools package** (https://github.com/BM32ESRF/lauetools). Author: J.-S. Micha, Oct 2026

# SETTINGS

In [ ]:
# 1. your experiment: RAW_DATA folder (at ESRF: /data/visitor/<proposal>/bm32/<date>/RAW_DATA)
RAW_DATA_FOLDER = '/data/visitor/<proposal>/bm32/<date>/RAW_DATA'
SCAN_FILTER = ''           # show only the maps whose name contains this text, e.g. 'ZONE3' ('': all maps)
SCAN_INDEX = None          # None: choose the map in the menu of STEP 1; or its index in the table, e.g. 1240
MODE = 'compute'           # 'compute': analyse the map (steps already done with the same parameters are reloaded)
                           # 'show': only show the results of a map analysed before (nothing computed, files unchanged)

# 2. your sample
KEY_MATERIAL = 'MgO'       # material to index (name in the LaueTools list: MW.find_material('Mg') prints the names with 'Mg')
NB_GRAINS = 5              # max nb of grains (orientations) to find in each image

# 3. calibration and parameters
DETFILE = 'auto'           # detector calibration .det file; 'auto': the most recent one of the experiment
PSPFILE = None             # optional: .psp file of the LaueTools GUI, its values are copied once into peaksearch.yaml
PEAKSEARCH = {'IntensityThreshold': 100, 'NumberMaxofFits': 20000, 'PeakSizeRange': [0.4, 10]}            # changes of peak search parameters, e.g. {'IntensityThreshold': 300}
                           # weak and broad spots: {'IntensityThreshold': 100, 'NumberMaxofFits': 20000, 'PeakSizeRange': [0.4, 10]}
INDEXING = {'central spots indices':np.arange(25).tolist(), 'NBMAXPROBED':30}              # changes of indexation parameters, e.g. {'e_max': 22, 'MinimumNumberMatches': 8}

# 4. computer
EXECUTION = 'auto'         # 'auto': cluster (SLURM) if available, else this computer; or 'local', 'slurm'
SLURM_MACHINE = 'magnifix3'     # cluster: 'auto', 'magnifix' (192 cpus), 'magnifix2', 'magnifix3' (2 or 3 nodes of 192 cpus),
                           # 'hpc6', 'hpc7', 'hpc8', 'nice' (96 cpus)
NB_CPUS = None             # None: all cpus of this computer, or of the cluster node

# 5. advanced
OUTPUT_ROOT = None         # None: results in PROCESSED_DATA (recommended); or another folder
REDO = False               # True: compute again the steps already done (not needed after a change of parameters: detected)

# IMPORTS and computer

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
try:
    import ipympl  # interactive plots (zoom, hover) in JupyterLab / VS Code
    %matplotlib widget
except ImportError:
    %matplotlib inline

import LaueTools.map_workflow as MW
import LaueTools.peaksearch_batch as PB
import LaueTools.indexing_batch as IB
import LaueTools.orientationclustering as OC
import LaueTools.generaltools as GT

execution = MW.environment(EXECUTION)

# STEP 1: choose a map

The table lists the 2D maps of the experiment (from the BLISS `.h5` file of `RAW_DATA_FOLDER`):
- `images`: image files found / expected (an interrupted scan has fewer images; `end` is then not `SUCCESS`)
- `done`: results already in `PROCESSED_DATA` (`peaks`: peak search, `fit:MgO`: indexation with MgO)

Choose the map in the menu below the table, then run the next cell.

In [ ]:
logfile = MW.find_logfile(RAW_DATA_FOLDER)
maps = MW.list_maps(logfile, contains=SCAN_FILTER, output_root=OUTPUT_ROOT)
MW.show_maps(maps)

In [ ]:
selector = MW.ScanSelector(maps, SCAN_INDEX)

In [ ]:
info = MW.scan_info(maps, selector.index, OUTPUT_ROOT)

print(f"map {info['index']}: {info['name']}")
print(f"  command   : {info['command']}")
print(f"  map       : {info['mapdims'][0]} x {info['mapdims'][1]} points, steps {info['stepsizes']} micron "
      f"(fast motor {info['fast_motor']}, slow motor {info['slow_motor']})")
print(f"  detector  : {info['CCDLabel']}, images {info['image_folder']}/{info['prefix']}{'#' * info['nbdigits']}{info['suffix']}")
print(f"  results in: {info['workdir']}")
if maps.loc[info['index'], 'done']:
    GT.printgreen(f"results exist for this map: {maps.loc[info['index'], 'done']}  (MODE = 'show' to only see them)")
if maps.loc[info['index'], 'end'] != 'SUCCESS':
    GT.printyellow(f"this scan ended with {maps.loc[info['index'], 'end']}: some images may be missing")

# STEP 2: settings files

The settings of the analysis are written in two YAML files in the results folder (printed below): `configs/peaksearch.yaml` and `configs/indexing.yaml`. They are the only reference of the parameters (with comments) and can be edited by hand (then run this cell again). Parameters of `PEAKSEARCH`, `INDEXING` and `PSPFILE` (SETTINGS) are written in these files each time this cell runs.

With `MODE = 'show'`, the files of the map are only read (unchanged).

**Check the calibration file**: it must be the one of the detector position of this experiment.

In [ ]:
show_only = MODE == 'show'
if show_only:
    pcfg, icfg = MW.make_configs(info, KEY_MATERIAL, None, update=False)
# calibration file
elif DETFILE == 'auto':
    detfiles = MW.find_detfiles(RAW_DATA_FOLDER)
    if not detfiles:
        raise FileNotFoundError('no .det calibration file found in the experiment: set DETFILE in SETTINGS')
    detfile = detfiles[0]
    print('calibration files of the experiment (most recent first):')
    for f in detfiles[:10]:
        print(f'   {"->" if f == detfile else "  "} {f}')
else:
    detfile = Path(DETFILE)

if not show_only:
    # peak search parameters: in peaksearch.yaml (defaults of the detector, then your changes)
    if PSPFILE is None:
        for f in MW.find_pspfiles(info['image_folder'])[:3]:
            print(f'(a .psp file of the LaueTools GUI exists for this scan: PSPFILE = {str(f)!r} copies its values in peaksearch.yaml)')
    else:
        print('values of', PSPFILE, 'copied in peaksearch.yaml')
    pcfg, icfg = MW.make_configs(info, KEY_MATERIAL, detfile, pspfile=PSPFILE, peaksearch=PEAKSEARCH,
                                 indexing={'nbGrainstoFind': NB_GRAINS, **INDEXING})
print()
PB.print_config(pcfg)
PB.check_config(pcfg)

# STEP 3: PEAK SEARCH

## test on one image

The parameters are printed first: their names can be used in `PEAKSEARCH` (SETTINGS). Look at the image: the green crosses should be on the Laue spots.
- spots without cross: decrease `IntensityThreshold`, e.g. `PEAKSEARCH = {'IntensityThreshold': 100}` in SETTINGS
- crosses in the noise, or more than ~1000 peaks: increase it

then run the SETTINGS cell, the cell of STEP 2 and this one again. Typical good images have 50 to 500 peaks.

In [ ]:
plt.close('all')   # figures of previous runs of the cells (interactive figures pile up in memory)
MW.print_parameters(pcfg, 'peaksearch')
TEST_IMAGE = None     # None: image of the middle of the map; or an image index

test_image = pcfg['run']['test_image_index'] if TEST_IMAGE is None else TEST_IMAGE
imagefile = PB.image_path(pcfg, test_image)
res = PB.peaksearch_image(imagefile, PB.params_from_config(pcfg, writefiles=False))
PB.print_result(res)

imagedata = PB.read_image(imagefile, pcfg['scan']['CCDLabel'])
ax = PB.plot_peaks(imagedata, res[2], pcfg['exclusion_boxes'], title=imagefile.name)
plt.show()

## all images

Long computations: a progress bar shows the progress (on the cluster: status line updated every 30 s). On the cluster, if you interrupt the waiting (stop button), the job goes on: run the cell again to wait for its end (it is not submitted again).

In [ ]:
plt.close('all')   # figures of previous runs of the cells (interactive figures pile up in memory)
peaks = MW.run_step('peaksearch', pcfg, execution, nb_cpus=NB_CPUS, machine=SLURM_MACHINE, redo=REDO)

if peaks is not None:
    summary_peaks = PB.summarize_results(peaks, pcfg['scan']['mapdims'])
    MW.plot_map(summary_peaks['nbpeaks2D'], pcfg, 'nb of peaks')
    plt.show()

# STEP 4: INDEXATION

## test on one image

A good result: grain 0 with **at least ~20 indexed spots** and a **matching rate above ~30 %** (green). If not:
- check `KEY_MATERIAL` and the calibration file (STEP 2)
- more spots for the indexation (weak patterns): `INDEXING = {'MinimumNumberMatches': 6}`; larger angular tolerance: `INDEXING = {'AngleTolLUT': 0.5}`

The parameters are printed first: their names can be used in `INDEXING` (SETTINGS).

In [ ]:
MW.print_parameters(icfg, 'indexing')
iparams = IB.params_from_config(icfg, useinternalmultiprocessing=False, writefitfile=False)
res = IB.index_refine(IB.corfile_path(icfg, test_image), iparams)
IB.print_result(res)

## all images

In [ ]:
plt.close('all')   # figures of previous runs of the cells (interactive figures pile up in memory)
fits = MW.run_step('indexing', icfg, execution, nb_cpus=NB_CPUS, machine=SLURM_MACHINE, redo=REDO,
                   show_only=show_only)

if fits is not None:
    summary = IB.summarize_results(fits, icfg['scan']['mapdims'])
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.3))
    MW.plot_map(summary['Nbindexedimages2D'], icfg, 'grain 0: nb of indexed spots', ax=axes[0])
    MW.plot_map(summary['MRindexedimages2D'], icfg, 'grain 0: matching rate (%)', ax=axes[1])
    plt.show()

# STEP 5: MAPS

## quality of the indexation

All grains found in each image (`_g0.fit`, `_g1.fit` ...) are read; at each point, the **best grain** (most indexed spots) is used. `g0` is only the first grain found, not always the best one.

A point is **not reliable** (black in the next maps) if its best grain has too few indexed spots or a large mean pixel deviation (difference between measured and computed spot positions). The map *why masked* and the histograms show the reason and the thresholds (red lines). If many points are masked by a deviation just above the threshold, the threshold may be too strict for these data (e.g. calibration measured at another temperature): set `MAX_PIXEL_DEVIATION` below. Default thresholds: `configs/indexing.yaml` (`postprocess:` section).

In [ ]:
plt.close('all')   # figures of previous runs of the cells (interactive figures pile up in memory)
MIN_INDEXED_SPOTS = None     # None: MinimumNbIndexedSpots of indexing.yaml
MAX_PIXEL_DEVIATION = None   # None: LargestMeanPixelResidue of indexing.yaml

grains = MW.load_grains(icfg)
fig, mask = MW.quality_maps(grains, icfg, MIN_INDEXED_SPOTS, MAX_PIXEL_DEVIATION)
plt.show()

## orientation: inverse pole figure (IPF) maps

Color = crystal direction along the sample axis x, y or z (color key on the right), for the best grain of each point. Same color = same orientation. Black: points not reliable (see above).

In [ ]:
fig = MW.orientation_maps(grains, icfg, mask=mask)
plt.show()

## grains (segmentation)

Grains are groups of neighbouring points with a misorientation smaller than `GRAIN_THRESHOLD` (deg), using all grains found in each image. Then:
- **KAM** (kernel average misorientation): local misorientation between neighbours (deformation, sub-grain boundaries)
- **GROD** (grain reference orientation deviation): misorientation to the reference orientation of the grain

In [ ]:
plt.close('all')   # figures of previous runs of the cells (interactive figures pile up in memory)
GRAIN_THRESHOLD = 1.0    # deg, larger: fewer grains
MIN_GRAIN_SIZE = 10      # min nb of points of a grain

result, analyzer = MW.segmentation(icfg, threshold=GRAIN_THRESHOLD, min_cluster_size=MIN_GRAIN_SIZE)
OC.print_cluster_summary(result, analyzer, nb_max=10)
fig, ax = MW.plot_clusters(result, analyzer, icfg, min_size=MIN_GRAIN_SIZE)
plt.show()

In [ ]:
kam = OC.compute_kam(analyzer, result, symmetry=result.symmetry)
grod = OC.compute_grod_map(result, analyzer)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.3))
MW.plot_map(kam, icfg, 'KAM (deg)', ax=axes[0], cmap='magma', vmin=0, vmax=np.nanpercentile(kam, 98))
MW.plot_map(grod, icfg, 'GROD (deg)', ax=axes[1], cmap='magma', vmin=0, vmax=np.nanpercentile(grod, 98))
plt.show()

## more

- strain, lattice parameters, misorientation to a reference point: `../indexation/2_postprocess_maps.ipynb`
- interactive browser of the grains (properties of each grain): `../indexation/3_segmentation_grains.ipynb`

with `CONFIG` = the indexation settings file printed below

In [ ]:
print("CONFIG = '%s'" % icfg['configfile'])

# STEP 6 (option): the same analysis for a series of maps

The maps of `SERIES` are analysed with the settings files of the map of STEP 1-4 (peak search, indexation), one after the other (each one on the cluster if available). Maps already done are skipped.

In [ ]:
SERIES = []    # scan indices of the table of STEP 1, e.g. [1240, 1259, 1278]
# or all the maps whose name contains a text:   SERIES = [i for i in maps.index if 'ZONE3' in maps.loc[i, 'dataset']]

series = {}
if SERIES:
    series = MW.process_maps(maps, SERIES, info, KEY_MATERIAL, detfile, output_root=OUTPUT_ROOT,
                             execution=execution, nb_cpus=NB_CPUS, machine=SLURM_MACHINE, redo=REDO)

In [ ]:
plt.close('all')   # figures of previous runs of the cells (interactive figures pile up in memory)
# nb of indexed spots (grain 0) of the maps of the series
if series:
    ncols = min(4, len(series))
    nrows = int(np.ceil(len(series) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.2 * ncols, 3.6 * nrows), squeeze=False)
    for ax, (index, cfg_i) in zip(axes.flat, series.items()):
        res_i = IB.load_allresults(MW._results_file('indexing', cfg_i))['dict_results_summary']
        MW.plot_map(res_i['Nbindexedimages2D'], cfg_i, f"{index}: {cfg_i['name']}", ax=ax)
    for ax in axes.flat[len(series):]:
        ax.axis('off')
    fig.tight_layout()
    plt.show()